<a href="https://colab.research.google.com/github/riyawakde16-art/Day29/blob/main/Day29.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

In [3]:
from google.colab import files
uploaded=files.upload()

Saving train.txt to train.txt


In [4]:
import pandas as pd
# Load the dataset
data = pd.read_csv(
    "train.txt",
    sep=";",
    names=["text", "emotions"])


In [5]:
from sklearn.preprocessing import LabelEncoder
# Create LabelEncoder object
encoder = LabelEncoder()
# Encode emotion labels
data["emotion_encoded"] = encoder.fit_transform(data["emotions"])
# Display mapping
mapping = dict(zip(encoder.classes_,
                   encoder.transform(encoder.classes_)))
print("Emotion Label Mapping:")
print(mapping)

Emotion Label Mapping:
{'anger': np.int64(0), 'fear': np.int64(1), 'joy': np.int64(2), 'love': np.int64(3), 'sadness': np.int64(4), 'surprise': np.int64(5)}


In [6]:
import string
import re
import nltk
from nltk.corpus import stopwords
# Download stopwords
nltk.download("stopwords")
# Load English stopwords
stop_words = set(stopwords.words("english"))
# Complete cleaning function
def clean_text(text):
    # Lowercase
    text = text.lower()
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    # Remove emojis & special characters
    text = text.encode("ascii", "ignore").decode()
    # Remove stopwords
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)
# Apply cleaning
data["cleaned_text"] = data["text"].apply(clean_text)
# Show original and cleaned text
print(data[["text", "cleaned_text"]].head(10))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


                                                text  \
0                            i didnt feel humiliated   
1  i can go from feeling so hopeless to so damned...   
2   im grabbing a minute to post i feel greedy wrong   
3  i am ever feeling nostalgic about the fireplac...   
4                               i am feeling grouchy   
5  ive been feeling a little burdened lately wasn...   
6  ive been taking or milligrams or times recomme...   
7  i feel as confused about life as a teenager or...   
8  i have been with petronas for years i feel tha...   
9                                i feel romantic too   

                                        cleaned_text  
0                              didnt feel humiliated  
1  go feeling hopeless damned hopeful around some...  
2          im grabbing minute post feel greedy wrong  
3  ever feeling nostalgic fireplace know still pr...  
4                                    feeling grouchy  
5      ive feeling little burdened lately wasnt sure 

In [7]:
# Save cleaned dataset
data.to_csv("cleaned_emotions.csv", index=False)
print("Cleaned dataset saved successfully as cleaned_emotions.csv")


Cleaned dataset saved successfully as cleaned_emotions.csv


Q.1

In [24]:
# Load cleaned Emotions dataset
data = pd.read_csv("cleaned_emotions.csv")
print("\nDataset:")
print(data.head())
print("\nColumn names:")
print(data.columns)
# Remove missing values
data = data.dropna(subset=["text", "emotions"])
# Separate features and target
X = data["text"]
y = data["emotions"]
print("\nNumber of records:", len(data))


Dataset:
                                                text emotions  \
0                            i didnt feel humiliated  sadness   
1  i can go from feeling so hopeless to so damned...  sadness   
2   im grabbing a minute to post i feel greedy wrong    anger   
3  i am ever feeling nostalgic about the fireplac...     love   
4                               i am feeling grouchy    anger   

   emotion_encoded                                       cleaned_text  
0                4                              didnt feel humiliated  
1                4  go feeling hopeless damned hopeful around some...  
2                0          im grabbing minute post feel greedy wrong  
3                3  ever feeling nostalgic fireplace know still pr...  
4                0                                    feeling grouchy  

Column names:
Index(['text', 'emotions', 'emotion_encoded', 'cleaned_text'], dtype='object')

Number of records: 16000


In [22]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [23]:
# Print shapes
print("\nX_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)


X_train shape: (12800,)
X_test shape : (3200,)
y_train shape: (12800,)
y_test shape : (3200,)


Q.2

In [12]:
# Create CountVectorizer
bow_vectorizer = CountVectorizer()
# Fit and transform training data
X_train_bow = bow_vectorizer.fit_transform(X_train)
# Transform testing data
X_test_bow = bow_vectorizer.transform(X_test)
# Print matrix shapes
print("\nX_train_bow shape:", X_train_bow.shape)
print("X_test_bow shape :", X_test_bow.shape)
# First 20 feature names
feature_names = bow_vectorizer.get_feature_names_out()
print("\nFirst 20 vocabulary words:")
print(feature_names[:20])


X_train_bow shape: (12800, 13501)
X_test_bow shape : (3200, 13501)

First 20 vocabulary words:
['aa' 'aaaaand' 'aaaand' 'aac' 'aahhh' 'aaron' 'ab' 'abandon' 'abandoned'
 'abandoning' 'abandonment' 'abated' 'abbigail' 'abc' 'abdomen'
 'abdominal' 'abducted' 'abhorrent' 'abide' 'abilities']


Q.3

In [13]:
# Create model
bow_model = MultinomialNB()
# Train model
bow_model.fit(X_train_bow, y_train)
# Predictions
y_pred_bow = bow_model.predict(X_test_bow)
# Accuracy
bow_accuracy = accuracy_score(y_test, y_pred_bow)
print("\nBag of Words Accuracy:")
print(bow_accuracy)
print("\nBag of Words Accuracy (%):")
print(f"{bow_accuracy * 100:.2f}%")



Bag of Words Accuracy:
0.7390625

Bag of Words Accuracy (%):
73.91%


Q.4

In [14]:
# Total vocabulary size
print("\nTotal vocabulary size:")
print(len(feature_names))
# Show any 15 words
print("\n15 vocabulary words:")
print(feature_names[:15])
# Convert one sample document into BoW vector
sample_document = X_train.iloc[0]
print("\nSample document:")
print(sample_document)
# Transform sample document
sample_vector = bow_vectorizer.transform([sample_document])
print("\nBoW vector:")
print(sample_vector)
print("\nBoW vector as array:")
print(sample_vector.toarray())



Total vocabulary size:
13501

15 vocabulary words:
['aa' 'aaaaand' 'aaaand' 'aac' 'aahhh' 'aaron' 'ab' 'abandon' 'abandoned'
 'abandoning' 'abandonment' 'abated' 'abbigail' 'abc' 'abdomen']

Sample document:
i refers of course though i cant help feeling somehow ironically in retrospect to loudons son with kate mcgarrigle the rather talented himself rufus wainwright

BoW vector:
<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 23 stored elements and shape (1, 13501)>
  Coords	Values
  (0, 1701)	1
  (0, 2622)	1
  (0, 4419)	1
  (0, 5504)	1
  (0, 5571)	1
  (0, 5931)	1
  (0, 6249)	1
  (0, 6514)	1
  (0, 7059)	1
  (0, 7338)	1
  (0, 8173)	1
  (0, 9521)	1
  (0, 9661)	1
  (0, 9928)	1
  (0, 10127)	1
  (0, 10995)	1
  (0, 11007)	1
  (0, 11751)	1
  (0, 11938)	1
  (0, 12005)	1
  (0, 12112)	1
  (0, 12951)	1
  (0, 13260)	1

BoW vector as array:
[[0 0 0 ... 0 0 0]]


Q.5

In [15]:
# Create CountVectorizer with unigrams + bigrams
bigram_vectorizer = CountVectorizer(
    ngram_range=(1, 2)
)
# Fit and transform training data
X_train_bigram = bigram_vectorizer.fit_transform(X_train)
# Transform testing data
X_test_bigram = bigram_vectorizer.transform(X_test)
# Print shape
print("\nBigram training matrix shape:")
print(X_train_bigram.shape)
print("\nBigram testing matrix shape:")
print(X_test_bigram.shape)
# Display some features
bigram_features = bigram_vectorizer.get_feature_names_out()
print("\nFirst 30 unigram + bigram features:")
print(bigram_features[:30])



Bigram training matrix shape:
(12800, 106150)

Bigram testing matrix shape:
(3200, 106150)

First 30 unigram + bigram features:
['aa' 'aa full' 'aa meeting' 'aaaaand' 'aaaaand tis' 'aaaand'
 'aaaand after' 'aac' 'aac or' 'aahhh' 'aahhh work' 'aaron' 'aaron has'
 'ab' 'abandon' 'abandon it' 'abandon me' 'abandon the' 'abandoned'
 'abandoned ask' 'abandoned believe' 'abandoned by' 'abandoning'
 'abandoning him' 'abandonment' 'abandonment has' 'abandonment to'
 'abated' 'abated and' 'abbigail']


Q.6

In [16]:
# Create MultinomialNB model
bigram_model = MultinomialNB()
# Train model
bigram_model.fit(X_train_bigram, y_train)
# Predictions
y_pred_bigram = bigram_model.predict(X_test_bigram)
# Accuracy
bigram_accuracy = accuracy_score(
    y_test,
    y_pred_bigram
)
print("\nBigram Accuracy:")
print(bigram_accuracy)
print("\nBigram Accuracy (%):")
print(f"{bigram_accuracy * 100:.2f}%")
# Compare with unigram accuracy
print("\nComparison:")
print(f"Unigram BoW Accuracy : {bow_accuracy * 100:.2f}%")
print(f"Bigram BoW Accuracy  : {bigram_accuracy * 100:.2f}%")



Bigram Accuracy:
0.6496875

Bigram Accuracy (%):
64.97%

Comparison:
Unigram BoW Accuracy : 73.91%
Bigram BoW Accuracy  : 64.97%


Q.7

In [17]:
# Create TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer()
# Fit and transform training data
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
# Transform testing data
X_test_tfidf = tfidf_vectorizer.transform(X_test)
# Print shapes
print("\nTF-IDF training matrix shape:")
print(X_train_tfidf.shape)
print("\nTF-IDF testing matrix shape:")
print(X_test_tfidf.shape)
# Feature names
tfidf_features = tfidf_vectorizer.get_feature_names_out()
# First 15 feature names
print("\nFirst 15 TF-IDF features:")
print(tfidf_features[:15])




TF-IDF training matrix shape:
(12800, 13501)

TF-IDF testing matrix shape:
(3200, 13501)

First 15 TF-IDF features:
['aa' 'aaaaand' 'aaaand' 'aac' 'aahhh' 'aaron' 'ab' 'abandon' 'abandoned'
 'abandoning' 'abandonment' 'abated' 'abbigail' 'abc' 'abdomen']


Q.8

In [18]:
# Create model
tfidf_model = MultinomialNB()
# Train model
tfidf_model.fit(X_train_tfidf, y_train)
# Predictions
y_pred_tfidf = tfidf_model.predict(X_test_tfidf)
# Accuracy
tfidf_accuracy = accuracy_score(
    y_test,
    y_pred_tfidf
)
print("\nTF-IDF Accuracy:")
print(tfidf_accuracy)
print("\nTF-IDF Accuracy (%):")
print(f"{tfidf_accuracy * 100:.2f}%")




TF-IDF Accuracy:
0.6175

TF-IDF Accuracy (%):
61.75%


Q.9

In [20]:
# Create comparison table
comparison = pd.DataFrame({
    "Method": [
        "Bag of Words (Unigrams)",
        "Bag of Words (Unigrams + Bigrams)",
        "TF-IDF"
    ],
    "Accuracy": [
        bow_accuracy,
        bigram_accuracy,
        tfidf_accuracy
    ]
})
# Accuracy in percentage
comparison["Accuracy (%)"] = (
    comparison["Accuracy"] * 100
).round(2)
print("\nComparison Table:")
print(comparison[["Method", "Accuracy (%)"]].to_string(index=False))
# Find best method
best_index = comparison["Accuracy"].idxmax()
best_method = comparison.loc[
    best_index,
    "Method"
]
best_accuracy = comparison.loc[
    best_index,
    "Accuracy"
]
print("\nBest Method:")
print(best_method)
print("\nBest Accuracy:")
print(f"{best_accuracy * 100:.2f}%")
# Short observation
print("\nObservation:")
print(
    f"{best_method} performed best with an accuracy of "
    f"{best_accuracy * 100:.2f}%. "
    "Different vectorization methods represent text differently, "
    "which can affect the performance of the Naive Bayes classifier."
)



Comparison Table:
                           Method  Accuracy (%)
          Bag of Words (Unigrams)         73.91
Bag of Words (Unigrams + Bigrams)         64.97
                           TF-IDF         61.75

Best Method:
Bag of Words (Unigrams)

Best Accuracy:
73.91%

Observation:
Bag of Words (Unigrams) performed best with an accuracy of 73.91%. Different vectorization methods represent text differently, which can affect the performance of the Naive Bayes classifier.


Q.10

In [21]:
# The three methods were already trained above.
# Now select the best method and save its vectorizer and model.

if best_method == "Bag of Words (Unigrams)":
    best_vectorizer = bow_vectorizer
    best_model = bow_model
    vectorizer_filename = "best_bow_vectorizer.joblib"
    model_filename = "best_bow_model.joblib"
elif best_method == "Bag of Words (Unigrams + Bigrams)":
    best_vectorizer = bigram_vectorizer
    best_model = bigram_model
    vectorizer_filename = "best_bigram_vectorizer.joblib"
    model_filename = "best_bigram_model.joblib"
else:
    best_vectorizer = tfidf_vectorizer
    best_model = tfidf_model
    vectorizer_filename = "best_tfidf_vectorizer.joblib"
    model_filename = "best_tfidf_model.joblib"
# Save best vectorizer
joblib.dump(
    best_vectorizer,
    vectorizer_filename
)
# Save best model
joblib.dump(
    best_model,
    model_filename
)
print("\nBest vectorizer saved as:")
print(vectorizer_filename)
print("\nBest model saved as:")
print(model_filename)
print("\nBest method:")
print(best_method)
print("\nBest accuracy:")
print(f"{best_accuracy * 100:.2f}%")


Best vectorizer saved as:
best_bow_vectorizer.joblib

Best model saved as:
best_bow_model.joblib

Best method:
Bag of Words (Unigrams)

Best accuracy:
73.91%

Assignment completed successfully!
